#### Package 설치

In [ ]:
# uv add docx2txt
# uv add langchain-upstage

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 키 값은 출력하지 않고 설정 여부만 확인
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
print("UPSTAGE_API_KEY 설정 확인 완료")

UPSTAGE_API_KEY 설정 확인 완료


#### 1. Knowledge Base 구성을 위한 데이터 생성

- [RecursiveCharacterTextSplitter](https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/)를 활용한 데이터 chunking
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
    - LangChain에서 다양한 [TextSplitter](https://python.langchain.com/v0.2/docs/how_to/#text-splitters)들을 제공
- `chunk_size` 는 split 된 chunk의 최대 크기
- `chunk_overlap`은 앞 뒤로 나뉘어진 chunk들이 얼마나 겹쳐도 되는지 지정

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('../data/tax_with_table.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

print(len(document_list))
print(type(document_list[0]))
print(document_list[:2])

C:\Users\user\AppData\Local\Temp\ipykernel_1392\1748737679.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


225
<class 'langchain_core.documents.base.Document'>
[Document(metadata={'source': '../data/tax_with_table.docx'}, page_content='소득세법\n\n소득세법\n\n[시행 2024. 1. 1.] [법률 제19933호, 2023. 12. 31., 일부개정]\n\n기획재정부(소득세제과(사업소득, 기타소득)) 044-215-4217\n\n기획재정부(소득세제과(근로소득)) 044-215-4216\n\n기획재정부(재산세제과(양도소득세)) 044-215-4314\n\n기획재정부(금융세제과(이자소득, 배당소득)) 044-215-4236\n\n\n\n\t제1장 총칙 <개정 2009. 12. 31.>\t\n\n\n\n제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다.\n\n[본조신설 2009. 12. 31.]\n\n[종전 제1조는 제2조로 이동 <2009. 12. 31.>]\n\n\n\n제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 12. 31.>\n\n1. “거주자”란 국내에 주소를 두거나 183일 이상의 거소(居所)를 둔 개인을 말한다.\n\n2. “비거주자”란 거주자가 아닌 개인을 말한다.\n\n3. “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다.\n\n4. “외국법인”이란 「법인세법」 제2조제3호에 따른 외국법인을 말한다.\n\n5. “사업자”란 사업소득이 있는 거주자를 말한다.\n\n② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다.\n\n[본조신설 2009. 12. 31.]\n\n\n\n제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무

In [2]:
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
print(embeddings)

client=<openai.resources.embeddings.Embeddings object at 0x000001A50E2EC4D0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001A50F369A60> model='solar-embedding-1-large' dimensions=None upstage_api_key=SecretStr('**********') upstage_api_base='https://api.upstage.ai/v1/solar' embedding_ctx_length=4096 embed_batch_size=10 allowed_special=set() disallowed_special='all' chunk_size=1000 max_retries=2 request_timeout=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers={'x-upstage-client': 'langchain'} default_query=None http_client=None http_async_client=None


In [3]:
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 저장 경로 (다른 노트북 인덱스와 섞이지 않도록 별도 경로 사용)
DB_PATH = "../db/faiss_docdb_docx"

# 데이터를 처음 저장할 때 
database = FAISS.from_documents(documents=document_list, embedding=embeddings)
# 로컬 파일로 저장
database.save_local(DB_PATH)

# 이미 저장된 데이터를 사용할 때 
# database = FAISS.load_local(DB_PATH, embeddings, allow_dangerous_deserialization=True)
print(database)

#### 2. 답변 생성을 위한 Retrieval

- `FAISS`에 저장한 데이터를 유사도 검색(`similarity_search()`)를 활용해서 가져옴

In [4]:
query = '비과세소득에는 어떤것들이 있나요?'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정
retrieved_docs = database.similarity_search(query, k=6)

print(len(retrieved_docs))
print(type(retrieved_docs[0]))
print(retrieved_docs[0].metadata)

6
<class 'langchain_core.documents.base.Document'>
{'source': '../data/tax_with_table.docx'}


In [5]:
print(retrieved_docs[0].page_content[:100])

마. 「국군포로의 송환 및 대우 등에 관한 법률」에 따라 국군포로가 받는 위로지원금과 그 밖의 금품

바. 「문화재보호법」에 따라 국가지정문화재로 지정된 서화ㆍ골동품의 양도로 발생


#### 3. Augmentation을 위한 Prompt 활용

- Retrieval된 데이터는 LangChain에서 제공하는 프롬프트(`"rlm/rag-prompt"`) 사용
- [LangChain Hub https://smith.langchain.com/hub/](https://smith.langchain.com/hub/) 

In [6]:

from langchain_upstage import ChatUpstage

print("===> 6. 생성 → LLM으로 답변 생성")
llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm)

===> 6. 생성 → LLM으로 답변 생성
metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}} client=<openai.resources.chat.completions.completions.Completions object at 0x000001A57ABB7C80> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001A52A702D80> root_client=<openai.OpenAI object at 0x000001A510110560> root_async_client=<openai.AsyncOpenAI object at 0x000001A57ABB6D80> model_name='solar-pro3' temperature=0.5 model_kwargs={} stream_chunk_timeout=120.0 disabled_params={'parallel_tool_calls': None} upstage_api_key=SecretStr('**********') upstage_api_base='https://api.upstage.ai/v1'


In [ ]:
# try:
    # from langchain_classic import hub
    # rag_prompt = hub.pull("rlm/rag-prompt")
    # print(" langchain-classic hub 작동")
# except ImportError:
#     print(" langchain-classic 미설치")

In [ ]:
from langsmith import Client

client = Client()

try:
    rag_prompt = client.pull_prompt(
        "rlm/rag-prompt",
        dangerously_pull_public_prompt=True,
    )
    print("hub 프롬프트 pull 성공")
    print(rag_prompt)
except ValueError as e:
    print(f"pull 실패(ValueError): {e}")

#### 4. 답변 생성

- [RetrievalQA](https://docs.smith.langchain.com/old/cookbook/hub-examples/retrieval-qa-chain)를 통해 LLM에 전달
    - `RetrievalQA`는 [create_retrieval_chain](https://python.langchain.com/v0.2/docs/how_to/qa_sources/#using-create_retrieval_chain)으로 대체됨
    - 실제 ChatBot 구현 시 `create_retrieval_chain`으로 변경하는 과정을 볼 수 있음

In [ ]:
from langchain_classic.chains import RetrievalQA  # langchain-classic 사용
from pprint import pprint

# 기존 코드 그대로 작동 (레거시 지원)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm, 
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": rag_prompt}
)

In [ ]:
query = '비과세소득에는 어떤것들이 있나요?'

ai_message = qa_chain.invoke({"query": query})
pprint(ai_message)

In [ ]:
query = '과세소득의 범위 및 소득의 구분에는 어떤것들이 있나요?'

ai_message = qa_chain.invoke({"query": query})
pprint(ai_message)

* LangChain 기반의 RAG(Retrieval-Augmented Generation) 파이프라인을 구현하여 DOCX 문서를 로드, 임베딩, 검색, 그리고 LLM을 통한 답변 생성

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter  # LCEL 호환
from langchain_core.documents import Document  #  최신 import
from langchain_core.runnables import RunnablePassthrough  #  LCEL
from langchain_core.output_parsers import StrOutputParser  #  LCEL
from langchain_core.prompts import PromptTemplate  #  LCEL
from langchain_community.document_loaders import Docx2txtLoader
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_classic import hub  #  hub는 classic으로

import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# 파이프라인 인덱스 저장 경로 (단계별 셀의 인덱스와 청크 설정이 달라 경로를 분리)
DB_PATH = "../db/faiss_docdb_docx_pipeline"

# 1. 환경 변수 로드
load_dotenv()

# 2. API 키 확인
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
if not UPSTAGE_API_KEY:
    raise ValueError("UPSTAGE_API_KEY 설정 필요 (.env)")

# 3. DOCX 로드 함수
def load_docx(file_path):
    loader = Docx2txtLoader(file_path)
    documents = loader.load()
    text = "\n".join([doc.page_content for doc in documents])
    if not text.strip():
        raise ValueError("문서 텍스트 없음")
    return text

# 4. 문서 분할
def split_text(text, chunk_size=500, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    return splitter.split_text(text)

# 5. 벡터스토어 생성
def create_vector_store(text_chunks, embedding_model):
    documents = [Document(page_content=chunk) for chunk in text_chunks]
    vector_store = FAISS.from_documents(documents, embedding_model)
    vector_store.save_local(DB_PATH)
    return vector_store

# 6. LCEL RAG 체인 (RetrievalQA 대체!)
def create_rag_chain(llm, retriever, prompt):
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

# 7. 질문 응답 함수
def query_with_llm(query, vector_store):
    llm = ChatUpstage(model="solar-pro", temperature=0.5)
    
    # Hub 프롬프트 (classic 사용)
    rag_prompt = hub.pull("rlm/rag-prompt")
    
    retriever = vector_store.as_retriever(search_kwargs={"k": 6})
    rag_chain = create_rag_chain(llm, retriever, rag_prompt)
    
    answer = rag_chain.invoke(query)
    return answer

# 실행
if __name__ == "__main__":
    docx_path = "../data/tax_with_table.docx"
    
    # 1. 문서 로드
    text = load_docx(docx_path)
    print(" 문서 로드 완료")
    
    # 2. 분할
    text_chunks = split_text(text)
    print(f" {len(text_chunks)}개 청크 생성")
    
    # 3. 임베딩
    embedding_model = UpstageEmbeddings(model="solar-embedding-1-large")
    
    # 4. 벡터스토어
    vector_store = create_vector_store(text_chunks, embedding_model)
    print(" 벡터스토어 생성 완료")
    
    # 5. 질의
    query = "총수입금액 불산입에 대하여 설명해 주세요."
    answer = query_with_llm(query, vector_store)
    
    print("\n AI 답변:")
    print(answer)

In [ ]:
from pprint import pprint

pprint(answer)

###  개선된 Source Level1

* uv add unstructured[docx]

In [ ]:
import re
import os
import warnings

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_upstage import UpstageEmbeddings, ChatUpstage

warnings.filterwarnings("ignore", category=UserWarning)

# ── 설정 (값을 바꿀 때 이곳만 수정) ──────────────────
DOCX_PATH = "../data/tax_with_table.docx"
DB_PATH = "../db/faiss_docdb_docx_level1"   # 다른 셀의 인덱스와 분리
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200
TOP_K = 7                                    # 최종적으로 LLM 에 전달할 문서 수
VECTOR_WEIGHT = 0.7                          # 하이브리드 검색에서 벡터 검색 점수의 비중 (나머지는 키워드)
EMBEDDING_MODEL = "solar-embedding-1-large"
CHAT_MODEL = "solar-pro"

# 환경 변수 로드 및 API 키 확인 (키 값은 출력하지 않음)
load_dotenv()
if not os.getenv("UPSTAGE_API_KEY"):
    raise ValueError("UPSTAGE_API_KEY 가 설정되지 않았습니다. .env 파일을 확인하세요.")


# ── 1. 문서 로드 ───────────────────────────────────
def load_docx(file_path: str) -> str:
    """DOCX 파일을 읽어 하나의 문자열로 반환합니다. (표 구조까지 읽기 위해 Unstructured 사용)"""
    documents = UnstructuredWordDocumentLoader(file_path).load()
    text = "\n".join(doc.page_content for doc in documents)
    if not text.strip():
        raise ValueError("문서에서 텍스트를 추출할 수 없습니다.")
    return re.sub(r"\n\s*\n", "\n\n", text)  # 연속된 빈 줄을 하나로 정리


# ── 2. 문서 분할 ───────────────────────────────────
def split_text(text: str) -> list[str]:
    """조문("제N조") 경계를 우선해서 청크로 분할합니다."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        # 앞쪽 구분자부터 시도: 조문 → 문단 → 줄 → 문장 → 단어
        separators=["\n제", "\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(text)


# ── 3. 벡터 저장소 생성 ────────────────────────────
def create_vector_store(chunks: list[str], embeddings) -> tuple:
    """청크에 메타데이터(chunk_id, 조문 번호)를 붙여 FAISS 에 저장합니다."""
    documents = []
    for i, chunk in enumerate(chunks):
        metadata = {"chunk_id": i}  # 하이브리드 검색에서 같은 청크를 구분하는 용도
        article = re.search(r"제\d+조", chunk)  # 청크에 처음 나오는 조문 번호
        if article:
            metadata["article"] = article.group()
        documents.append(Document(page_content=chunk, metadata=metadata))

    vector_store = FAISS.from_documents(documents, embeddings)
    vector_store.save_local(DB_PATH)
    return vector_store, documents


# ── 4. 검색 (벡터 + 키워드) ────────────────────────
def keyword_search(query: str, documents: list[Document], k: int) -> list[Document]:
    """질문의 단어가 청크에 많이 포함될수록 높은 점수를 주는 단순 키워드 검색."""
    words = query.split()
    # 단어가 청크 안에 포함되는지로 비교 (한국어 조사가 붙어도 일부 일치)
    return sorted(documents, key=lambda d: sum(w in d.page_content for w in words), reverse=True)[:k]


def hybrid_search(query: str, vector_store, documents: list[Document], k: int = TOP_K) -> list[Document]:
    """벡터 검색과 키워드 검색 결과를 순위 점수로 합쳐 상위 k개를 반환합니다."""
    vector_docs = vector_store.similarity_search(query, k=k * 2)
    keyword_docs = keyword_search(query, documents, k=k * 2)

    scores = {}      # chunk_id → 합산 점수
    docs_by_id = {}  # chunk_id → 문서
    for docs, weight in ((vector_docs, VECTOR_WEIGHT), (keyword_docs, 1 - VECTOR_WEIGHT)):
        for rank, doc in enumerate(docs):
            chunk_id = doc.metadata["chunk_id"]
            docs_by_id[chunk_id] = doc
            # 순위가 높을수록 큰 점수 (1등 = weight), 두 검색에 모두 나온 청크는 점수가 합쳐짐
            scores[chunk_id] = scores.get(chunk_id, 0) + weight * (1 - rank / len(docs))

    top_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return [docs_by_id[chunk_id] for chunk_id in top_ids]


# ── 5. 프롬프트와 답변 생성 ────────────────────────
PROMPT_TEMPLATE = """당신은 한국 세법 전문가입니다. 주어진 법률 문서를 바탕으로 정확하고 자세한 답변을 제공해야 합니다.

다음 규칙을 반드시 따르세요:
1. 법조문의 조항, 항, 호, 목을 정확히 인용하세요
2. 전문 용어를 사용할 때는 쉬운 설명을 함께 제공하세요
3. 관련 조항들 간의 연관성을 설명하세요
4. 실무적 적용 방법도 함께 설명하세요
5. 불확실한 내용이 있으면 명시적으로 언급하세요

참고 문서:
{context}

질문: {question}

위 법률 문서를 바탕으로 정확하고 자세한 답변을 제공해주세요. 관련 조항을 인용하며 설명해주세요."""


def query_with_llm(query: str, vector_store, documents: list[Document]) -> dict:
    """하이브리드 검색으로 문서를 찾고, LLM 으로 답변을 생성합니다."""
    print(f"질의: {query}")
    print("하이브리드 검색 수행 중...")
    relevant_docs = hybrid_search(query, vector_store, documents)
    print(f"검색된 관련 문서: {len(relevant_docs)}개")

    # 검색된 청크에 [문서 N] 번호를 붙여 하나의 문자열로 합침
    context = "\n\n".join(f"[문서 {i}]\n{doc.page_content}" for i, doc in enumerate(relevant_docs, 1))
    messages = ChatPromptTemplate.from_template(PROMPT_TEMPLATE).format_messages(context=context, question=query)

    print("LLM 응답 생성 중...")
    llm = ChatUpstage(model=CHAT_MODEL, temperature=0.5)
    response = llm.invoke(messages)
    return {"answer": response.content, "source_documents": relevant_docs, "context_used": context}


# ── 6. 검색 품질 평가 ──────────────────────────────
def evaluate_context_quality(query: str, retrieved_docs: list[Document]) -> float:
    """검색된 문서의 품질을 0~1 로 평가합니다. (질문 단어 포함 비율 70% + 문서 길이 30%)"""
    words = query.split()
    if not words or not retrieved_docs:
        return 0.0
    scores = []
    for doc in retrieved_docs:
        keyword_match = sum(w in doc.page_content for w in words) / len(words)
        length_score = min(len(doc.page_content) / 1000, 1.0)  # 1000자 이상이면 만점
        scores.append(keyword_match * 0.7 + length_score * 0.3)
    return sum(scores) / len(scores)


# ── 실행 예제 ──────────────────────────────────────
if __name__ == "__main__":
    print("=== 개선된 RAG 파이프라인 실행 ===\n")

    print("1. 문서 로드 중...")
    text = load_docx(DOCX_PATH)
    print(f"   문서 로드 완료: {len(text):,} 문자\n")

    print("2. 문서 분할 중...")
    text_chunks = split_text(text)
    print(f"   문서 분할 완료: {len(text_chunks)}개 청크 생성\n")

    print("3. 임베딩 모델 초기화 및 벡터 저장소 생성 중...")
    embedding_model = UpstageEmbeddings(model=EMBEDDING_MODEL)
    vector_store, documents = create_vector_store(text_chunks, embedding_model)
    print("   벡터 저장소 생성 완료\n")

    print("4. 질의 실행 중...")
    query = "총수입금액 불산입에 대하여 설명해 주세요."
    results = query_with_llm(query, vector_store, documents)
    print("\n답변:")
    print(results["answer"])

In [ ]:
from pprint import pprint

pprint(results)

In [ ]:
# 6. 컨텍스트 품질 평가
context_quality = evaluate_context_quality(query, results["source_documents"])

# 7. 결과 출력
print("\n" + "="*60)
print(" AI의 답변:")
print("="*60)
print(results["answer"])

print("\n" + "="*60)
print(" 검색 결과 요약:")
print("="*60)
print(f"• 참고한 문서 조각 수: {len(results['source_documents'])}개")
print(f"• 컨텍스트 품질 점수: {context_quality:.2f}/1.00")
print(f"• 총 컨텍스트 길이: {len(results['context_used']):,} 문자")

print("\n" + "="*60)
print("📄 참고한 문서 미리보기:")
print("="*60)
for i, doc in enumerate(results["source_documents"][:3]):  # 상위 3개만 미리보기
    preview = doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content
    print(f"\n[문서 {i+1}] {preview}")
print("="*60)